# verses

> a printed verse number read, verse runs mended and renumbered, verses assembled into chapters, Devanagari akṣaras repaired

In [ ]:
#| default_exp verses

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import re
from difflib import SequenceMatcher
from fastcore.all import AttrDict, L

A verse is `{n, text, page, heading}`. A page is `{page, blocks}`, where each block is `{kind, n, text}`: what `vishalakshi.read_doc` keeps from a vision model's reading. Every function that puts a number right notes what it did in `probs`, a list the caller passes in, for a proofreader to check.

## Verse numbers

`vnum` reads a printed verse number in ASCII, Devanagari or Tamil digits. Tamil numbers written with the old signs ௰ ten, ௱ hundred and ௲ thousand compose (௬௱௨ is 602). Of two numbers printed together it takes the `last` and notes it: `२६॥२७` is two verses read as one. `pick='first'` suits a print that sets a running number before a count within its part (`२६॥ १`).

In [ ]:
#| export
_DIG = str.maketrans('०१२३४५६७८९௦௧௨௩௪௫௬௭௮௯', '01234567890123456789')
_SIGNS = {'௰': 10, '௱': 100, '௲': 1000}
def _tamil(s):
    "A run of digits and the old Tamil signs ௰ ten, ௱ hundred, ௲ thousand as an int: ௬௱௨ is 602, ௨௱௩ 203."
    tot, d = 0, None
    for c in s:
        if c.isdecimal(): d = (d or 0) * 10 + int(c)
        else: tot += (d or 1) * _SIGNS[c]; d = None
    return tot + (d or 0)

def vnum(n, page=None, probs:list=None, pick:str='last') -> int|None:
    "A printed verse number as an int, or None when it has no digits; of several printed together, the `last` or the `first`."
    xs = [_tamil(x) for x in re.findall(r'[\d௰௱௲]+', str('' if n is None else n).translate(_DIG))]
    if len(xs) > 1 and probs is not None: probs.append(f'{page}: verses {"-".join(map(str, xs))} printed as one, numbered {xs[-1 if pick == "last" else 0]}')
    return (xs[-1] if pick == 'last' else xs[0]) if xs else None

In [ ]:
_p = []
test_eq(vnum('॥ २६ ॥'), 26); test_eq(vnum('(௪௦)'), 40); test_eq(vnum(12), 12); test_eq(vnum(None), None)
test_eq(vnum('२६॥२७', 'p5', _p), 27); test_eq(_p, ['p5: verses 26-27 printed as one, numbered 27'])
test_eq(vnum('௬௱௨'), 602); test_eq(vnum('௨௱௩'), 203); test_eq(vnum('௰'), 10)
test_eq(vnum('२६॥ १', pick='first'), 26)   # a running number, then the count within its part

## Mending a run

`mend` puts right two slips a reader makes, where the neighbours prove them: a number read twice where the one before is missing (148, 150, 150: the first is 149), and two verses read as one where a number is missing between them, split after the inner line that closes a verse. `pair`, for a book of two-line verses whose halves both close on a daṇḍa, cuts where the two parts are nearest in length. `renumber` mends a long chapter's numbers: a run read at a constant offset that the sequence then returns from (53-58 read 63-68, then 59), and a number read without its leading digit (101 as 1).

In [ ]:
#| export
_END = re.compile(r'[.।॥|]\s*$')
def mend(vs, probs:list, pair:bool=False) -> list:
    "Verses with a doubled number and two verses read as one put right, where their neighbours prove it."
    out = []
    for i, v in enumerate(vs):
        n, prev = v['n'], (out[-1]['n'] if out else 0)
        nxt = vs[i + 1]['n'] if i + 1 < len(vs) else None
        if n == nxt and prev == n - 2:
            out.append(dict(v, n=n - 1)); probs.append(f"{v['page']}: first of two {n}s taken as {n - 1}"); continue
        ls = v['text'].split('\n')
        cut = [j for j, l in enumerate(ls[:-1]) if _END.search(l) and j >= (1 if pair else 2)]
        if prev == n - 2 and cut:
            j = (min(cut, key=lambda c: abs(len(''.join(ls[:c + 1])) - len(''.join(ls[c + 1:])))) if pair else cut[-1]) + 1
            out += [dict(v, n=n - 1, text='\n'.join(ls[:j])), dict(v, text='\n'.join(ls[j:]), heading=None)]
            probs.append(f"{v['page']}: {n} split after its line {j}, the first part taken as {n - 1}"); continue
        out.append(v)
    return out

def renumber(vs, where, probs:list) -> list:
    "Verse numbers a long chapter's print makes hard to read, put right from their neighbours."
    vs = [dict(v) for v in vs]
    for i in range(1, len(vs)):
        want = vs[i - 1]['n'] + 1; d = vs[i]['n'] - want
        if not d: continue
        j = i
        while j < len(vs) and j - i < 15 and vs[j]['n'] - d == want + (j - i): j += 1
        if i < j < len(vs) and vs[j]['n'] == want + (j - i):
            for k in range(i, j): probs.append(f"{where}: verse read as {vs[k]['n']} numbered {vs[k]['n'] - d} ({vs[k]['page']})"); vs[k]['n'] -= d
    out = []
    for i, v in enumerate(vs):
        n = v['n']; want = out[-1]['n'] + 1 if out else n
        nxt = vs[i + 1]['n'] if i + 1 < len(vs) else None
        k = 10 ** len(str(n))
        if n != want and (nxt == want + 1 or (n < want and want % k == n % k)):
            probs.append(f"{where}: verse read as {n} numbered {want} ({v['page']})"); n = want
        out.append(dict(v, n=n))
    return out

In [ ]:
V = lambda n, t='a ।\nb ॥', pg=1: dict(n=n, text=t, page=pg, heading=None)
_p = []
test_eq([v['n'] for v in mend([V(148), V(150), V(150), V(151)], _p)], [148, 149, 150, 151]); test_eq(len(_p), 1)
_p = []
_m = mend([V(1), V(3, 'a\nb\nc ॥\nd\ne ॥')], _p)
test_eq([(v['n'], v['text']) for v in _m], [(1, 'a ।\nb ॥'), (2, 'a\nb\nc ॥'), (3, 'd\ne ॥')])
_p = []
test_eq([v['n'] for v in renumber([V(n) for n in (52, 63, 64, 65, 56, 57)], 'ch 3', _p)], [52, 53, 54, 55, 56, 57]); test_eq(len(_p), 3)
test_eq([v['n'] for v in renumber([V(n) for n in (99, 100, 1, 102)], 'ch 9', [])], [99, 100, 101, 102])

## Chapters

`assemble` gathers verses from pages in order. An unnumbered fragment carries onto the next numbered verse (a verse begun on one page and closed on the next). A heading names the chapter it opens; a verse numbered 1 after a higher number starts a new chapter; a colophon closes one, taking an unnumbered verse just before it as the chapter's last. The same verse read twice (a spread photographed twice) is kept once. Each chapter's run is then mended. Only `kinds` count as verse (`verse`, or a block with no kind); `kinds=None` takes every block but headings and colophons, for a reading that labels verse text by its own kinds. A commentary a reading labels by its own kind (an urai, a gloss, a bhāṣya) belongs to its verse: name the kinds in `notes` and each goes into its verse's `notes`, under its kind. Some prints number the commentary, not the verse (the Vidhānamālai sets the number at the urai's end): `number_from_note` gives the verse that number. It returns the chapters and the problems seen.

In [ ]:
#| export
def _alike(a, b, at=.8): return SequenceMatcher(None, a, b).ratio() >= at

def assemble(pages, pair:bool=False, kinds=('verse', None), notes=(), number_from_note:bool=False) -> tuple:
    """Chapters `[{heading, colophon, verses}]` from pages of blocks, and the problems seen. `kinds` None takes every block but headings and
    colophons as verse. A block of a kind in `notes` (`urai`, `gloss`, `bhashya`) goes into its verse's `notes`; with `number_from_note`, a
    numbered note gives the unnumbered verse before it that number."""
    chs, cur, buf, probs, hd, pend = [], None, [], [], None, {}
    new = lambda h=None: dict(heading=h, colophon=None, verses=[])
    def add(n, text, pg):
        nonlocal cur, hd, pend
        if cur is None or (cur['verses'] and n == 1 and cur['verses'][-1]['n'] > 1):
            if cur and cur['verses']: chs.append(cur)
            cur = new()
        if cur['verses'] and n != cur['verses'][-1]['n'] + 1: probs.append(f"{pg}: verse {n} after {cur['verses'][-1]['n']}")
        v = dict(n=n, text=text, page=pg, heading=hd)
        if pend: v['notes'], pend = pend, {}
        cur['verses'].append(v); hd = None
        return v
    for p in pages:
        for b in p.get('blocks') or p.get('items') or []:
            k, t, n, pg = b.get('kind'), b.get('text') or '', b.get('n'), p.get('page')   # text as read: a leading break or an empty block is the reader's
            if k in notes:
                m = vnum(n, pg, probs) if n is not None else None
                if number_from_note and buf and m is not None:
                    add(m, '\n'.join(buf), pg); buf = []; probs.append(f'{pg}: verse numbered {m} from its {k}')
                last = cur['verses'][-1] if cur and cur['verses'] and not buf else None
                (last.setdefault('notes', {}) if last else pend).setdefault(k, []).append(t)
                continue
            if k == 'start':   # a chapter opens here: `repairs.cut_by_plan`
                if cur and cur['verses']: chs.append(cur)
                cur = None; continue
            if k == 'heading':
                hd = t
                if cur is None: cur = new(t)
                elif not cur['heading']: cur['heading'] = t
                continue
            if k == 'colophon':
                if buf and cur and cur['verses']:
                    n = cur['verses'][-1]['n'] + 1
                    cur['verses'].append(dict(n=n, text='\n'.join(buf), page=pg, heading=None)); buf = []
                    probs.append(f'{pg}: unnumbered verse before the colophon taken as {n}')
                (cur or new())['colophon'] = t
                if cur: chs.append(cur); cur = None
                continue
            if kinds is not None and k not in kinds: continue
            if (n := vnum(n, pg, probs) if n is not None else None) is None:
                if not (buf and _alike(buf[-1], t)): buf.append(t)
                continue
            if cur and any(v['n'] == n and _alike(v['text'].split('\n')[-1], t.split('\n')[-1], .6) for v in cur['verses']): hd = None; continue
            add(n, '\n'.join(buf + [t]), pg); buf = []
    if cur and cur['verses']: chs.append(cur)
    if buf: probs.append(f'unfinished verse at the end: {buf[-1][:40]}')
    for c in chs: c['verses'] = mend(c['verses'], probs, pair)
    return chs, probs

def check_counts(chs, counts) -> list:
    "Where the chapters' verse counts differ from the counts a plan expects."
    out = [] if len(chs) == len(counts) else [f'{len(chs)} chapters, the plan has {len(counts)}']
    return out + [f"chapter {i + 1}: {len(c['verses'])} verses, the plan has {k}" for i, (c, k) in enumerate(zip(chs, counts)) if len(c['verses']) != k]

In [ ]:
B = lambda kind, text, n=None: dict(kind=kind, text=text, n=n)
_pages = [dict(page=0, blocks=[B('heading', 'अथ प्रथमोऽध्यायः'), B('verse', 'क ।\nख ॥१॥', '१'), B('verse', 'ग ।', None)]),
          dict(page=1, blocks=[B('verse', 'घ ॥२॥', '॥२॥'), B('verse', 'ङ ॥३॥', 3), B('colophon', 'इति प्रथमोऽध्यायः')]),
          dict(page=2, blocks=[B('verse', 'च ॥१॥', 1), B('verse', 'छ ॥२॥', 2), B('verse', 'छ ॥२॥', 2), B('prose', 'an English note')])]
_chs, _pr = assemble(_pages)
test_eq(len(_chs), 2); test_eq(_chs[0]['heading'], 'अथ प्रथमोऽध्यायः'); test_eq(_chs[0]['colophon'], 'इति प्रथमोऽध्यायः')
test_eq([(v['n'], v['text']) for v in _chs[0]['verses']], [(1, 'क ।\nख ॥१॥'), (2, 'ग ।\nघ ॥२॥'), (3, 'ङ ॥३॥')])
test_eq([v['n'] for v in _chs[1]['verses']], [1, 2]); test_eq(_pr, [])
test_eq(check_counts(_chs, [3, 2]), []); test_eq(check_counts(_chs, [3, 3]), ['chapter 2: 2 verses, the plan has 3'])

In [ ]:
_u = [dict(page=0, blocks=[B('verse', 'பா', 1), B('urai', 'உரை', 1)])]
test_eq(len(assemble(_u)[0][0]['verses']), 1); test_eq([v['text'] for v in assemble(_u, kinds=None)[0][0]['verses']], ['பா', 'உரை'])   # every kind but headings and colophons

In [ ]:
_vm = [dict(page=0, blocks=[B('verse', 'பா ஒன்று', None), B('urai', 'உரை ஒன்று', 1), B('verse', 'பா இரண்டு', None), B('urai', 'உரை இரண்டு', 2)])]
_c, _p = assemble(_vm, notes=('urai',), number_from_note=True)
test_eq([(v['n'], v['text'], v['notes']['urai']) for v in _c[0]['verses']], [(1, 'பா ஒன்று', ['உரை ஒன்று']), (2, 'பா இரண்டு', ['உரை இரண்டு'])])
_g = [dict(page=0, blocks=[B('verse', 'a ॥१॥', 1), B('gloss', 'its gloss'), B('verse', 'b ॥२॥', 2)])]
test_eq(assemble(_g, notes=('gloss',))[0][0]['verses'][0]['notes'], {'gloss': ['its gloss']})
test_eq('notes' in assemble(_g)[0][0]['verses'][0], False)   # without `notes` a gloss is not a verse and is not kept

## Akṣaras

`AKSHARA_BAD` names the malformed akṣaras a typed or converted Devanagari source carries; `bad_aksharas` lists which a text holds. `fix_aksharas` mends the mechanical ones: an i-sign typed after a virāma moved past its consonant cluster (श्िच → श्चि), a virāma before another sign or an anusvāra dropped, ट्ः plus a vowel letter read as ठ with that vowel, a stray nukta dropped, e + ā typed for o joined, a vowel letter with a sign typed after it made one letter, a doubled sign made one. The Taittirīya's own ग्ं stays.

In [ ]:
#| export
_C = '[\u0915-\u0939\u0958-\u095f]'   # consonants, the nukta letters too
_SIGN = {'अ': '', 'आ': 'ा', 'इ': 'ि', 'ई': 'ी', 'उ': 'ु', 'ऊ': 'ू', 'ए': 'े', 'ऐ': 'ै', 'ओ': 'ो', 'औ': 'ौ'}
AKSHARA_BAD = {'virāma before a vowel sign': re.compile('्[ा-ौॢॣ]'),
               'two vowel signs': re.compile('[\u093e-\u0944\u0946-\u094c][\u093e-\u0944\u0946-\u094c]'),
               'virāma before anusvāra or visarga': re.compile('(?<!ग)्[ंः]'),
               'nukta after a vowel sign': re.compile('[ा-्]़'),
               'vowel sign after a vowel letter': re.compile('[ऄ-औ][ा-ौ]')}

def bad_aksharas(t:str) -> list:
    "The names of the malformed akṣaras in `t`."
    return [k for k, r in AKSHARA_BAD.items() if r.search(t or '')]

def fix_aksharas(t:str) -> str:
    "`t`'s Devanagari with its mechanical slips mended."
    if not t: return t
    t = re.sub(f'्ि((?:{_C}्)*{_C})', lambda m: '्' + m[1] + 'ि', t)
    t = re.sub('ट्ः([अआइईउऊएऐओऔ])', lambda m: 'ठ' + _SIGN[m[1]], t)
    t = re.sub('्([ा-ौॢॣ])', r'\1', t)
    t = re.sub('(?<!ग)्([ं])', r'\1', t)
    t = re.sub('([ा-्])़', r'\1', t)
    t = t.replace('\u0947\u093e', '\u094b').replace('\u0948\u093e', '\u094c').replace('\u094d\u0903', '\u094d')
    for a, b in (('अा', 'आ'), ('आा', 'आ'), ('अो', 'ओ'), ('अौ', 'औ'), ('अे', 'ए'), ('एे', 'ऐ'), ('अि', 'इ'), ('इि', 'इ'), ('अी', 'ई'), ('इी', 'ई'), ('उू', 'ऊ'),
                 ('\u0941\u0942', '\u0942'), ('\u0940\u093f', '\u0940'), ('\u0940\u0947', '\u0940')): t = t.replace(a, b)
    return re.sub(r'([ा-ौ])\1', r'\1', t)

In [ ]:
test_eq(fix_aksharas('श्िच'), 'श्चि'); test_eq(fix_aksharas('भक्ित'), 'भक्ति'); test_eq(fix_aksharas('ध्ार्म'), 'धार्म')
test_eq(fix_aksharas('कण्ट्ःए'), 'कण्ठे'); test_eq(fix_aksharas('अाम'), 'आम'); test_eq(fix_aksharas('ग्ं'), 'ग्ं')
test_eq(bad_aksharas('ध्ार्म'), ['virāma before a vowel sign']); test_eq(bad_aksharas(fix_aksharas('ध्ार्म')), [])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()